# Цифровая обработка сигналов. Часть 2
## Лабораторная работа №1. Частотный конвертер
Выполнила: Перминова Анастасия, группа М4121

In [25]:
# импортируем нужные нам библиотеки
import numpy as np
from scipy.signal import lfilter, iirfilter, firwin
from math import gcd
from IPython.display import Audio, display
import librosa

# высчитываем коэффициенты повышения (L) и понижения (M) частоты дискретизации
# это нужно, чтобы дальше работать с целыми числами

def coeff_resample(fs_src, fs_tgt):  # fs_src - исходная частота, fs_tgt - целевая частота
    g = gcd(int(fs_src), int(fs_tgt))  # gcd - наибольший общий делитель
    return fs_tgt // g, fs_src // g  # (L, M)

# растягиваем сигнал, добавляя нулевые значения между значениями исходного сигнала

def upsample(x, L):
    if L == 1:
        return x.copy()
    y = np.zeros(len(x) * L, dtype=x.dtype)
    y[::L] = x
    return y

# прореживаем: выбираем из сигнала M-ые отсчеты

def downsample(x, M):
    return x[::M] if M > 1 else x.copy()

# ФИЛЬТРЫ
# КИХ-фильтр

def design_fir(M, ntaps=501, beta=8.6):  # ntaps - длина фильтра (методом подбора, пока не улучшился результат)
    if M <= 1:
        return np.array([1.0])
    cutoff = 1.0 / M  # частота среза
    return firwin(ntaps, cutoff, window=('kaiser', beta))

# БИХ-фильтр

def design_iir(M, order=4):  # order определяет, насколько крутым будет срез (определялся подбором)
    if M <= 1:
        return np.array([1.0]), np.array([1.0])
    cutoff = 1.0 / M
    return iirfilter(order, cutoff, btype='lowpass', ftype='butter')

# сам частотный конвертер
# три этапа: растяжение >> фильтрация >> прореживание

def freq_convert(x, fs_src, fs_tgt, method="fir"):
    L, M = coeff_resample(fs_src, fs_tgt)
    x_up = upsample(x, L)
    if method == "fir":
        h = design_fir(M)
        y_filt = lfilter(h, [1.0], x_up)
    else:
        b, a = design_iir(M, order=4)
        y_filt = lfilter(b, a, x_up)
    return downsample(y_filt, M)

'''
При дальнейшей реализации функций получилось так, что при понижении частоты
результаты были не хуже librosa, однако при повышении звучание было куда хуже,
поэтому попробовала другую версию КИХ-фильтра, как мне показалось, звук при повышении
получился получше
'''

def design_fir2(x, fs_src, fs_tgt):

    if fs_tgt <= fs_src:  # используем только при повышении частоты
        raise ValueError("fs_tgt должен быть больше fs_src для повышения частоты")

    L, M = coeff_resample(fs_src, fs_tgt)
    x_up = upsample(x, L)

    ntaps = 3001  # очень длинный фильтр
    cutoff = 1.0 / M
    h = firwin(numtaps=ntaps, cutoff=cutoff, window='blackmanharris')  # выбран другой тип окна (подавляет боковые лепестки)
    h /= np.sum(h)  # чтобы сохранить амплитуду после фильтрации

    y_filt = np.convolve(x_up, h, mode='same')  # свертка: нули заменяются на интерполированные значения, спектр.копии удаляются

    # прореживание
    y_out = downsample(y_filt, M)

    # нормировка выходного сигнала
    y_out /= np.max(np.abs(y_out)) + 1e-12

    return y_out

# реализация librosa

def baseline_resample(x, fs_src, fs_tgt):
    return librosa.resample(x, orig_sr=fs_src, target_sr=fs_tgt, res_type='kaiser_best')

# загружаем наш аудиофайл

file = 'arcade.wav'
x_full, fs_full = librosa.load(file, sr=None, mono=True)
print(f'Loaded {file}, fs = {fs_full}')

start = 5.0  # начинаем с 5 секунды
duration = 2.0  # длительность 2 секунды
x = x_full[int(start*fs_full): int((start+duration)*fs_full)]
print(f'Segment length: {len(x)} samples ({len(x)/fs_full:.2f} s)')

# воспроизводим исходный сигнал

print('Original signal:')
display(Audio(x, rate=fs_full))

# сравниваем частотные конвертеры

cases = [
    (44100, 8000),  # понижение частоты
    (16000, 44100)  # повышение частоты
]

# преобразуем сигнал к нужной частоте

for fs_src, fs_tgt in cases:
    print(f'\n====== {fs_src} → {fs_tgt} Hz ======')
    if fs_src != fs_full:
        x_src = baseline_resample(x, fs_full, fs_src)
    else:
        x_src = x.copy()

    # FIR конвертер
    if fs_tgt > fs_src:
        y_fir = design_fir2(x_src, fs_src, fs_tgt)
    else:
        y_fir = freq_convert(x_src, fs_src, fs_tgt, method='fir')
    print('FIR:')
    display(Audio(y_fir, rate=fs_tgt))

    # IIR конвертер
    y_iir = freq_convert(x_src, fs_src, fs_tgt, method='iir')
    print('IIR:')
    display(Audio(y_iir, rate=fs_tgt))

    # реализация librosa
    y_base = baseline_resample(x_src, fs_src, fs_tgt)
    print('Librosa:')
    display(Audio(y_base, rate=fs_tgt))

print('\nThe end.')


Loaded arcade.wav, fs = 44100
Segment length: 88200 samples (2.00 s)
Original signal:



====== 44100 → 8000 Hz ======
FIR:


IIR:


Librosa:



====== 16000 → 44100 Hz ======
FIR:


IIR:


Librosa:



The end.


При понижении частоты получилось сравниться с librosa, однако при повышении частоты librosa справилась куда лучше. Хотя улучшенный FIR показал результат приятнее первоначального, однако слышны помехи, при IIR же и вовсе слишком выделяются "режущие" звуки.